# Notebook 2: train, track, register

Train a few classifiers, log everything to MLflow, pick the best run, register it as `BankChurnModel`, set the `champion` alias.

When this finishes, serve with `mlflow run . -e serve` and try `app/gradio_ui.py`.

Notebook 1 first if you want the EDA context, but it's not required.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import seaborn as sns
from mlflow import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

## Configuration

Paths, experiment name, feature lists. Categoricals get one-hot encoded; numerics get scaled.

In [3]:
DEMO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = DEMO_ROOT / "data" / "bank_customers.csv"
MLFLOW_DB = DEMO_ROOT / "mlflow.db"

EXPERIMENT_NAME = "bank-churn"
MODEL_NAME = "BankChurnModel"
CHAMPION_ALIAS = "champion"

CATEGORICAL_COLS = ["country", "gender"]
NUMERIC_COLS = [
    "credit_score", "age", "tenure", "balance",
    "products_number", "credit_card", "active_member", "estimated_salary",
]
FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS

## MLflow tracking

Local SQLite backend (`mlflow.db` in the project root). Browse runs later with `mlflow ui`.

In [4]:
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
mlflow.set_experiment(EXPERIMENT_NAME)

2026/06/26 16:10:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/26 16:10:20 INFO mlflow.store.db.utils: Updating database tables
2026/06/26 16:10:21 INFO mlflow.tracking.fluent: Experiment with name 'bank-churn' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/dmin/Development/PyCharmProjects/ai-course-event-demo/notebooks/mlruns/1', creation_time=1782479421124, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782479421124, lifecycle_stage='active', name='bank-churn', tags={}, trace_location=None, workspace='default'>

## Train / test split

80/20 split. `stratify=y` keeps the churn ratio roughly the same in both sets.

In [5]:
df = pd.read_csv(DATA_PATH)
X = df[FEATURE_COLS]
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

## Dataset logging

Attach the training set to each run. `input_example` is just for reference; autolog handles signatures.

In [6]:
train_dataset = mlflow.data.from_pandas(X_train.assign(churn=y_train), name="training_set")
input_example = X.head(3)

## Preprocessing pipeline

One sklearn `Pipeline`: preprocess, then classify. Same object gets logged and served, so inference uses the same transforms as training.

In [7]:
def make_pipeline(estimator):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLS),
            ("num", StandardScaler(), NUMERIC_COLS),
        ])),
        ("classifier", estimator),
    ])

## Autolog

Turn off generic autolog, turn on sklearn autolog so each run saves the fitted pipeline and its signature.

In [8]:
mlflow.autolog(disable=True)
mlflow.sklearn.autolog(
    log_models=True,
    log_input_examples=False,
    log_model_signatures=True,
    log_datasets=False,
)

## Evaluation helpers

For churn, accuracy is the wrong thing to obsess over. We log precision, recall, F1, ROC-AUC, PR-AUC, plus confusion matrix / ROC / PR plots as artifacts.

In [9]:
def evaluate_classifier(y_true, y_pred, y_proba):
    return {
        "test_accuracy": float((y_true == y_pred).mean()),
        "test_precision": precision_score(y_true, y_pred, zero_division=0),
        "test_recall": recall_score(y_true, y_pred, zero_division=0),
        "test_f1_score": f1_score(y_true, y_pred, zero_division=0),
        "test_roc_auc": roc_auc_score(y_true, y_proba),
        "test_pr_auc": average_precision_score(y_true, y_proba),
    }

In [10]:
def log_eval_plots(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set(xlabel="Predicted", ylabel="True", title="Confusion matrix")
    fig.tight_layout()
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"ROC (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curve")
    ax.legend()
    fig.tight_layout()
    mlflow.log_figure(fig, "roc_curve.png")
    plt.close(fig)

    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    ap = average_precision_score(y_true, y_proba)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision, label=f"PR (AP={ap:.3f})")
    ax.set(xlabel="Recall", ylabel="Precision", title="Precision–recall curve")
    ax.legend()
    fig.tight_layout()
    mlflow.log_figure(fig, "pr_curve.png")
    plt.close(fig)

In [11]:
def train_and_log(run_name, pipeline, tags=None):
    """Fit, score on the test set, log metrics and plots."""
    tags = tags or {}
    with mlflow.start_run(run_name=run_name):
        for k, v in tags.items():
            mlflow.set_tag(k, v)
        mlflow.log_input(train_dataset, context="training")
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        metrics = evaluate_classifier(y_test, y_pred, y_proba)
        mlflow.log_metrics(metrics)
        log_eval_plots(y_test, y_pred, y_proba)
        mlflow.log_text(
            classification_report(y_test, y_pred, target_names=["No churn", "Churn"], zero_division=0),
            "classification_report.txt",
        )
        print(f"{run_name}: f1={metrics['test_f1_score']:.3f}, roc_auc={metrics['test_roc_auc']:.3f}")
        return metrics

## Baseline

Logistic regression with `class_weight="balanced"`. Everything else should beat this.

In [12]:
train_and_log(
    "baseline-logistic",
    make_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    tags={"stage": "baseline", "model_type": "LogisticRegression"},
)

2026/06/26 16:10:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


baseline-logistic: f1=0.499, roc_auc=0.777


{'test_accuracy': 0.7135,
 'test_precision': 0.38722826086956524,
 'test_recall': 0.7002457002457002,
 'test_f1_score': 0.49868766404199477,
 'test_roc_auc': 0.7771654551315569,
 'test_pr_auc': 0.46791921719038454}

## Model comparison

A few Random Forest and XGBoost configs. Each one is its own MLflow run.

In [13]:
for cfg in [
    {"n_estimators": 100, "max_depth": 8},
    {"n_estimators": 200, "max_depth": 12},
    {"n_estimators": 300, "max_depth": 15},
]:
    train_and_log(
        f"rf-n{cfg['n_estimators']}-d{cfg['max_depth']}",
        make_pipeline(RandomForestClassifier(**cfg, class_weight="balanced", random_state=42, n_jobs=-1)),
        tags={"stage": "comparison", "model_type": "RandomForest"},
    )

# XGBoost uses scale_pos_weight instead of class_weight for imbalance.
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
for cfg in [
    {"n_estimators": 100, "max_depth": 4, "learning_rate": 0.1},
    {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.05},
]:
    train_and_log(
        f"xgb-n{cfg['n_estimators']}-d{cfg['max_depth']}-lr{cfg['learning_rate']}",
        make_pipeline(XGBClassifier(**cfg, scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42)),
        tags={"stage": "comparison", "model_type": "XGBoost"},
    )

2026/06/26 16:10:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


rf-n100-d8: f1=0.603, roc_auc=0.864


2026/06/26 16:10:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


rf-n200-d12: f1=0.622, roc_auc=0.859


2026/06/26 16:10:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/26 16:10:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


rf-n300-d15: f1=0.614, roc_auc=0.854


2026/06/26 16:10:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


xgb-n100-d4-lr0.1: f1=0.606, roc_auc=0.870
xgb-n200-d6-lr0.05: f1=0.609, roc_auc=0.856


## Compare runs

Sorted by test F1. `mlflow ui` in a terminal if you want to click around.

In [14]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_f1_score DESC"],
)

display_cols = ["tags.mlflow.runName", "metrics.test_f1_score", "metrics.test_roc_auc", "metrics.test_recall"]
print(runs_df[display_cols].head(10))

  tags.mlflow.runName  metrics.test_f1_score  metrics.test_roc_auc  \
0         rf-n200-d12               0.621714              0.859244   
1         rf-n300-d15               0.613527              0.854479   
2  xgb-n200-d6-lr0.05               0.609148              0.855897   
3   xgb-n100-d4-lr0.1               0.605882              0.869624   
4          rf-n100-d8               0.602629              0.864078   
5   baseline-logistic               0.498688              0.777165   

   metrics.test_recall  
0             0.668305  
1             0.624079  
2             0.719902  
3             0.759214  
4             0.732187  
5             0.700246  


## Pick the champion

Top run by F1. In a real project you'd probably argue about the metric.

In [15]:
METRIC = "test_f1_score"
best_run = runs_df.iloc[0]
best_run_id = best_run["run_id"]
best_run_name = best_run.get("tags.mlflow.runName", best_run_id[:8])
best_metric = best_run[f"metrics.{METRIC}"]

print(f"Best run: {best_run_name} ({METRIC}={best_metric:.4f})")

Best run: rf-n200-d12 (test_f1_score=0.6217)


## Register the model

Promote the best run to the Model Registry as `BankChurnModel`. That's what the `serve` entry point in `MLproject` loads.

In [16]:
result = mlflow.register_model(model_uri=f"runs:/{best_run_id}/model", name=MODEL_NAME)

client = MlflowClient()
client.update_model_version(
    name=MODEL_NAME,
    version=result.version,
    description=f"Champion from run '{best_run_name}' ({METRIC}={best_metric:.4f})",
)

Successfully registered model 'BankChurnModel'.
2026/06/26 16:10:40 WARNING mlflow.tracking._model_registry.fluent: Run with id 0531a5663f3148b5a1d15f682164839c has no artifacts at artifact path 'model', registering model based on models:/m-4cba1ccbc08e41b4b4acf1691b7bb35b instead
Created version '1' of model 'BankChurnModel'.


<ModelVersion: aliases=[], creation_timestamp=1782479440312, current_stage='None', deployment_job_state=None, description="Champion from run 'rf-n200-d12' (test_f1_score=0.6217)", last_updated_timestamp=1782479440321, metrics=None, model_id=None, name='BankChurnModel', params=None, run_id='0531a5663f3148b5a1d15f682164839c', run_link=None, source='models:/m-4cba1ccbc08e41b4b4acf1691b7bb35b', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

## Champion alias

Point `champion` at the new version. Serving uses `models:/BankChurnModel@champion`, so you can promote without touching deploy config.

In [17]:
client.set_registered_model_alias(name=MODEL_NAME, alias=CHAMPION_ALIAS, version=result.version)